# Raising Exceptions in Python: 18 NEW Advanced Problems
## A step-by-step, explanation-first tutorial with fully worked solutions

**How this is different from the previous problem set:** these are **18 new scenarios**, organized like the provided lesson: introduce one idea, pause to predict, run a small code cell, explain what Python did, extend the implementation, and verify the result with executable assertions. Each problem has multiple short Markdown explanations rather than one giant answer cell.

**Scope from the supplied notebook:** `raise`, valid exception classes/instances, `.args`, propagation, bare `raise`, exception translation, implicit context, explicit causes, `from None`, and tracebacks. The later chapters extend those ideas to descriptors, generators, async code, thread pools, exception groups, and realistic parsers. Extensions are identified as such.

**Runtime:** Python **3.11 or newer**, standard library only. Run cells top-to-bottom. Every deliberately failing call is enclosed in an appropriate handler; successful execution finishes without an uncaught exception.

### Two accuracy notes before we begin

1. The supplied lesson says that `raise` requires an *instance*. Python also allows an exception **class** derived from `BaseException`: `raise ValueError` creates an instance automatically. If that class requires constructor arguments, raising the class alone will fail during instantiation. An unrelated class or ordinary object cannot be raised.
2. `raise NewError(...) from original` establishes an **explicit cause**; it does *not* graft the original traceback onto the new exception. `raise NewError(...) from None` suppresses presentation of implicit context, but `new_error.__context__` can still hold the original object. To attach a traceback explicitly, use `.with_traceback(tb)`, with care.

We will prove these claims with tests, not rely on wording alone.

In [1]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from functools import wraps
import re
import struct
import sys
import traceback
import unittest

assert sys.version_info >= (3, 11), 'This tutorial requires Python 3.11+.'


def capture(expected, call, *args, **kwargs):
    """Return the caught expected exception, or fail loudly for a wrong outcome."""
    try:
        call(*args, **kwargs)
    except expected as error:
        return error
    raise AssertionError(f'Expected {expected!r} but the call succeeded')


print('Environment:', sys.version.split()[0], '| tutorial helpers ready')


Environment: 3.13.7 | tutorial helpers ready


### Reading this notebook

Every chapter contains the **problem contract**, a **prediction checkpoint**, three incremental implementation/observation steps, a set of **executable acceptance checks**, and a **why-it-works conclusion**. The questions are immediately followed by worked reasoning, so you can cover the explanation and try it yourself first. The code is deliberately divided into small, readable cells.

---

# Part I — What exactly is raised?

The first problems take the source lesson's basic `raise` statement into APIs with constructor requirements and state invariants. Pay special attention to *when* an exception is constructed and to the difference between invalid types and invalid values.

## Problem 01 — The exception class that cannot be raised without arguments

**Scenario.** A library exposes an error type whose constructor requires a machine-readable code. A colleague writes `raise ProvisioningError`, expecting Python to fill in the code.

### Contract and constraints

Define a custom exception with a required `code` and optional detail. Demonstrate the failure of raising the bare class and the correct way to raise an initialized object. Distinguish the actual error caused by instantiation from the intended error.

### Predict before running

Will `raise ProvisioningError` raise a `ProvisioningError`, a `TypeError`, or something else? What is the content of `.args` when construction succeeds?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Define a real exception with a required constructor argument

In [2]:
class ProvisioningError(Exception):
    def __init__(self, code, detail=''):
        self.code = code
        self.detail = detail
        super().__init__(f'{code}: {detail}' if detail else code)

print(issubclass(ProvisioningError, Exception))


True


**Why this step matters.** Subclassing `Exception` makes the class a legitimate exception *type*. It does not make its constructor arguments optional. `super().__init__` gives the exception a useful one-element `.args` payload.

### Step 2 — Observe automatic class instantiation and its limit

In [3]:
def raise_uninitialized():
    raise ProvisioningError

construction_failure = capture(TypeError, raise_uninitialized)
print(type(construction_failure).__name__, str(construction_failure))


TypeError ProvisioningError.__init__() missing 1 required positional argument: 'code'


**Why this step matters.** Python attempts to instantiate an exception class operand with no arguments. The required `code` is missing, so **construction** raises `TypeError` before a `ProvisioningError` instance exists.

### Step 3 — Construct the intended failure at the point of use

In [4]:
def provision(kind):
    if kind not in {'cpu', 'storage'}:
        raise ProvisioningError('UNKNOWN_KIND', f'unsupported: {kind!r}')
    return f'allocated {kind}'

failed_provision = capture(ProvisioningError, provision, 'network')
print(failed_provision.code, failed_provision.args)
print(provision('cpu'))


UNKNOWN_KIND ("UNKNOWN_KIND: unsupported: 'network'",)
allocated cpu


**Why this step matters.** An exception instance can carry both stable structured data (`code`) and a readable message. The normal path returns without raising.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [5]:
assert 'code' in str(construction_failure)
assert failed_provision.code == 'UNKNOWN_KIND'
assert failed_provision.args == ("UNKNOWN_KIND: unsupported: 'network'",)
assert isinstance(failed_provision, Exception)
assert provision('storage') == 'allocated storage'
print('Problem 01: all checks passed')


Problem 01: all checks passed


### Complete explanation and design takeaway

Valid `raise` operands include an exception instance **or** an exception class. The class form is practical only when a no-argument constructor makes sense. Domain errors with required fields should normally be instantiated explicitly so call sites supply meaningful data.

**Your turn — extension question.** What happens if you write `raise ProvisioningError("E01")` with no detail?

**Reasoned answer.** The optional detail defaults to an empty string, so it raises `ProvisioningError` with `.code == "E01"` and `.args == ("E01",)`. This does not involve the problematic no-argument class form.

---

## Problem 02 — A property setter that must never corrupt valid state

**Scenario.** A temperature controller accepts a limit between 1 and 100. Failed assignments must leave the previous setting intact.

### Contract and constraints

Write a property-backed class. Reject booleans and non-integers with `TypeError`, reject out-of-range integers with `ValueError`, then update state only after every check passes.

### Predict before running

If the initial limit is 20 and you attempt to assign 0, should the stored value change? Why should `True` not silently mean 1?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Separate the storage location from the property contract

In [6]:
class ThermalController:
    def __init__(self, limit):
        self._limit = None
        self.limit = limit

    @property
    def limit(self):
        return self._limit

    @limit.setter
    def limit(self, value):
        # Validation will be inserted here in Step 2.
        self._limit = value

controller = ThermalController(20)
print('Starting limit:', controller.limit)


Starting limit: 20


**Why this step matters.** A property setter centralizes validation for both initialization and later assignments. The first draft deliberately lacks validation; the next cell replaces the class before it is used for meaningful work.

### Step 2 — Raise before committing the state transition

In [7]:
class ThermalController:
    def __init__(self, limit):
        self._limit = None
        self.limit = limit

    @property
    def limit(self):
        return self._limit

    @limit.setter
    def limit(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError('limit must be an integer, not bool or another type')
        if not 1 <= value <= 100:
            raise ValueError('limit must be between 1 and 100')
        self._limit = value

controller = ThermalController(20)
print('Valid state:', controller.limit)


Valid state: 20


**Why this step matters.** Only the final line mutates state. A raised exception exits the setter first. The explicit boolean check matters because `isinstance(True, int)` is true.

### Step 3 — Deliberately fail several assignments and inspect state

In [8]:
for bad in (0, 101, True, '25'):
    expected = TypeError if isinstance(bad, bool) or not isinstance(bad, int) else ValueError
    error = capture(expected, setattr, controller, 'limit', bad)
    print(repr(bad), '->', type(error).__name__, '| current:', controller.limit)
controller.limit = 42
print('Committed valid setting:', controller.limit)


0 -> ValueError | current: 20
101 -> ValueError | current: 20
True -> TypeError | current: 20
'25' -> TypeError | current: 20
Committed valid setting: 42


**Why this step matters.** The failed attempts are contained by `capture`, so the notebook keeps running. Both failure category and state integrity are part of the contract.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [9]:
assert controller.limit == 42
for value, expected in [(False, TypeError), (1.0, TypeError), (-3, ValueError), (200, ValueError)]:
    before = controller.limit
    capture(expected, setattr, controller, 'limit', value)
    assert controller.limit == before
controller.limit = 100
assert controller.limit == 100
print('Problem 02: all checks passed')


Problem 02: all checks passed


### Complete explanation and design takeaway

The best time to `raise` is **before** a transition that would violate an invariant. Use `TypeError` for the wrong kind of object and `ValueError` for an unacceptable value of the right kind. The design keeps successful and failed operations distinguishable.

**Your turn — extension question.** How would validation change if the limit were allowed to be any real number, but not `NaN`?

**Reasoned answer.** Accept the documented numeric types, reject booleans explicitly, then use `math.isfinite(value)` to reject `NaN` and infinities before the range check. Be clear about whether `Decimal` and third-party numeric types are supported.

---

## Problem 03 — A nested configuration failure with a useful path and a real cause

**Scenario.** A configuration loader must read `service.timeout` and convert a user-supplied string to an integer. Its clients need a stable path field when parsing fails.

### Contract and constraints

Expose `ConfigPathError(path, message)` with a tuple-valued path. Translate failed integer parsing with `raise ... from error`. Handle missing keys separately so callers can distinguish missing and malformed data.

### Predict before running

For `{"service": {"timeout": "tomorrow"}}`, which object should be the public exception? What should its `__cause__` contain?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Represent the location as structured data

In [10]:
class ConfigPathError(Exception):
    def __init__(self, path, message):
        self.path = tuple(path)
        super().__init__(f'{".".join(self.path)}: {message}')

sample = ConfigPathError(('service', 'timeout'), 'invalid')
print(sample.path, str(sample))


('service', 'timeout') service.timeout: invalid


**Why this step matters.** Store a field path as a tuple; callers can programmatically identify the failing location without parsing display text. The message remains useful for humans.

### Step 2 — Build a thin boundary around lookup and conversion

In [11]:
def read_timeout(config):
    try:
        raw = config['service']['timeout']
    except KeyError as error:
        raise ConfigPathError(('service', str(error.args[0])), 'missing key') from error
    if not isinstance(raw, str):
        raise ConfigPathError(('service', 'timeout'), 'expected a string')
    try:
        parsed = int(raw)
    except ValueError as error:
        raise ConfigPathError(('service', 'timeout'), 'not an integer') from error
    if not 1 <= parsed <= 3600:
        raise ConfigPathError(('service', 'timeout'), 'out of range')
    return parsed

print(read_timeout({'service': {'timeout': '45'}}))


45


**Why this step matters.** The narrow handlers translate only known failures. Explicit `from error` connects a low-level parser error to the public configuration error. Domain-range failure is raised directly: no lower-level failure occurred.

### Step 3 — Inspect the public contract and the underlying parser

In [12]:
malformed_timeout = capture(ConfigPathError, read_timeout, {'service': {'timeout': 'tomorrow'}})
missing_timeout = capture(ConfigPathError, read_timeout, {'service': {}})
print(malformed_timeout.path, type(malformed_timeout.__cause__).__name__)
print(missing_timeout.path, type(missing_timeout.__cause__).__name__)


('service', 'timeout') ValueError
('service', 'timeout') KeyError


**Why this step matters.** The outer error describes the consumer-facing operation; its explicit cause documents which lower-level exception originally failed. For an absent timeout, the cause is `KeyError`; for bad numeric text it is `ValueError`.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [13]:
assert read_timeout({'service': {'timeout': '1'}}) == 1
assert read_timeout({'service': {'timeout': '3600'}}) == 3600
assert malformed_timeout.path == ('service', 'timeout')
assert isinstance(malformed_timeout.__cause__, ValueError)
assert isinstance(missing_timeout.__cause__, KeyError)
range_error = capture(ConfigPathError, read_timeout, {'service': {'timeout': '0'}})
assert range_error.__cause__ is None
print('Problem 03: all checks passed')


Problem 03: all checks passed


### Complete explanation and design takeaway

Exception translation belongs at an abstraction boundary where the caller needs a new vocabulary. `from error` sets `__cause__` and formats a causal chain; it does not splice the old traceback into the new exception object. Typed path metadata removes brittle string parsing.

**Your turn — extension question.** What additional validation is needed for `{"service": 5}`?

**Reasoned answer.** The current contract demonstrates `KeyError` and parsing failures, but indexing an integer raises `TypeError`. To make the boundary complete, check that `config` and `config["service"]` are mappings before indexing and raise an appropriately located `ConfigPathError` directly. Do not simply catch every possible exception.

---

# Part II — Which exception reaches the caller?

These problems zoom in on propagation, `except` handler boundaries, `else`, context, and cause. They deliberately avoid broad `except Exception` wrappers that can accidentally hide bugs.

## Problem 04 — A regular-expression API with two distinct failure categories

**Scenario.** A search function receives a regular expression and a document. An invalid expression is a configuration failure; a valid expression that finds no match is a normal domain failure.

### Contract and constraints

Translate only `re.error` to `PatternSetupError` using an explicit cause. Raise `LookupError` directly if a valid pattern has no match. Do not confuse parser errors with absence of results.

### Predict before running

If the pattern is `"["`, should the caller see `LookupError` or `PatternSetupError`? What about `"z+"` on the string `"abc"`?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Introduce the public failure type and observe regex compilation

In [14]:
class PatternSetupError(Exception):
    pass

raw_pattern_failure = capture(re.error, re.compile, '[')
print(type(raw_pattern_failure).__name__)


PatternError


**Why this step matters.** The standard-library regex compiler already signals invalid syntax with `re.error`. We should map that one well-understood failure, not catch arbitrary programming errors.

### Step 2 — Write the API and make each raise intentional

In [15]:
def require_match(expression, text):
    if not isinstance(expression, str) or not isinstance(text, str):
        raise TypeError('expression and text must both be strings')
    try:
        pattern = re.compile(expression)
    except re.error as error:
        raise PatternSetupError(f'invalid expression: {expression!r}') from error
    match = pattern.search(text)
    if match is None:
        raise LookupError('pattern is valid but did not match')
    return match.group(0)

print(require_match(r'\d+', 'item 123'))


123


**Why this step matters.** The `try` block is as narrow as possible: only `re.compile` belongs to the syntax-translation boundary. A failed search is intentionally a separate exception with no fake compiler cause.

### Step 3 — Exercise the two failure branches

In [16]:
syntax_failure = capture(PatternSetupError, require_match, '[', 'abc')
no_match_failure = capture(LookupError, require_match, 'z+', 'abc')
print('Bad pattern cause:', type(syntax_failure.__cause__).__name__)
print('Missing match cause:', no_match_failure.__cause__)


Bad pattern cause: PatternError
Missing match cause: None


**Why this step matters.** The two inputs lead to different exceptions for different reasons, allowing a caller to react precisely.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [17]:
assert isinstance(syntax_failure.__cause__, re.error)
assert no_match_failure.__cause__ is None
assert require_match('a+', 'baaa!') == 'aaa'
assert isinstance(capture(TypeError, require_match, 5, 'text'), TypeError)
print('Problem 04: all checks passed')


Problem 04: all checks passed


### Complete explanation and design takeaway

A low-level error should be translated when it leaks an implementation-specific vocabulary. A legitimate domain failure should normally be raised directly, without an invented cause. This gives downstream handlers an actionable distinction.

**Your turn — extension question.** Should a catch-all `except Exception` replace `re.error` around the entire function?

**Reasoned answer.** No. That would disguise unrelated defects—perhaps a typo or a bug in future code—as a malformed regular expression. Catch only the exception the boundary is deliberately translating.

---

## Problem 05 — Add context in a decorator without replacing the exception

**Scenario.** You are building a service instrumentation decorator. A `LookupError` from the wrapped function must reach callers as the *same object*, but it should carry an extra diagnostic breadcrumb.

### Contract and constraints

Use an exception note (Python 3.11+) and bare `raise` inside a narrow handler. Show that identity, type, and original traceback remain available.

### Predict before running

If you write `raise LookupError(str(error))`, what identity and traceback information would be lost or altered?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Create a function that fails in a predictable location

In [18]:
def fetch_widget(widget_id):
    inventory = {'A': 10}
    if widget_id not in inventory:
        raise LookupError(f'unknown widget: {widget_id}')
    return inventory[widget_id]

original_widget_error = capture(LookupError, fetch_widget, 'B')
print(type(original_widget_error).__name__, str(original_widget_error))


LookupError unknown widget: B


**Why this step matters.** We already have a meaningful exception. Replacing it merely to add a breadcrumb would needlessly change object identity and potentially complicate diagnosis.

### Step 2 — Attach a note and re-raise the active object

In [19]:
def annotate_lookup(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except LookupError as error:
            error.add_note(f'while executing {fn.__name__}')
            raise
    return wrapper

annotated_fetch = annotate_lookup(fetch_widget)
print('Healthy request:', annotated_fetch('A'))


Healthy request: 10


**Why this step matters.** Bare `raise` propagates the active exception object. `add_note` adds human-readable context without mutating `.args` or changing its type. The decorator does not swallow the failure.

### Step 3 — Observe the exact exception before and after propagation

In [20]:
seen_in_function = []

def fail_with_known_instance():
    error = LookupError('sentinel missing')
    seen_in_function.append(error)
    raise error

annotated_failure = annotate_lookup(fail_with_known_instance)
propagated_error = capture(LookupError, annotated_failure)
print('Same object:', propagated_error is seen_in_function[0])
print('Notes:', propagated_error.__notes__)


Same object: True
Notes: ['while executing fail_with_known_instance']


**Why this step matters.** The identity assertion proves no new exception replaced the original; a note augments it. In contrast, `raise error` would add a new explicit-raise frame, even though it too would propagate the same object.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [21]:
assert propagated_error is seen_in_function[0]
assert propagated_error.args == ('sentinel missing',)
assert propagated_error.__notes__ == ['while executing fail_with_known_instance']
frames = traceback.extract_tb(propagated_error.__traceback__)
assert any(frame.name == 'fail_with_known_instance' for frame in frames)
assert annotated_fetch('A') == 10
print('Problem 05: all checks passed')


Problem 05: all checks passed


### Complete explanation and design takeaway

Instrumentation should generally enrich and re-raise known failures rather than manufacture replacements. In Python 3.11+, notes make the context visible in formatted tracebacks while preserving the underlying error’s type and arguments.

**Your turn — extension question.** Why does the decorator catch `LookupError` rather than `BaseException`?

**Reasoned answer.** Its documented responsibility is a lookup-related breadcrumb, not interception of interpreter-shutdown signals or cancellation. Catching broader exception families would create responsibilities the wrapper has not defined.

---

## Problem 06 — An exception raised inside one except block cannot jump to its sibling

**Scenario.** A developer expects a second sibling `except` clause to handle a new error raised by the first handler. You must demonstrate Python’s actual control flow and repair the design.

### Contract and constraints

Construct an inner `try` with two sibling handlers. Make the `ValueError` handler raise `RuntimeError`. Verify that only an *outer* handler catches the new error.

### Predict before running

Will the inner `except RuntimeError` execute after a `RuntimeError` is raised inside its neighboring `except ValueError`?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Record the order of handler execution

In [22]:
handler_events = []

def sibling_handlers():
    try:
        raise ValueError('bad input')
    except ValueError:
        handler_events.append('inner-value-handler')
        raise RuntimeError('failure while handling input')
    except RuntimeError:
        handler_events.append('inner-runtime-handler')

outer_error = capture(RuntimeError, sibling_handlers)
print(handler_events)


['inner-value-handler']


**Why this step matters.** Python selects one matching handler for the original exception. A fresh exception created inside that handler propagates *out* of the current `try` statement; sibling handlers are not a second search pass.

### Step 2 — Add a real outer boundary

In [23]:
def repaired_handler_flow():
    events = []
    try:
        try:
            raise ValueError('bad input')
        except ValueError:
            events.append('translate')
            raise RuntimeError('translated problem')
        except RuntimeError:
            events.append('unreachable-sibling')
    except RuntimeError as error:
        events.append('outer-recovery')
        return events, error

flow_events, translated = repaired_handler_flow()
print(flow_events, type(translated.__context__).__name__)


['translate', 'outer-recovery'] ValueError


**Why this step matters.** The outer `try` is a new, enclosing exception boundary. Because `RuntimeError` was raised while handling `ValueError`, Python implicitly records the original as `__context__`.

### Step 3 — Contrast implicit context with an explicit cause

In [24]:
def deliberate_translation():
    try:
        raise ValueError('invalid row')
    except ValueError as error:
        raise RuntimeError('row processing failed') from error

explicit_translation = capture(RuntimeError, deliberate_translation)
print(type(explicit_translation.__cause__).__name__, explicit_translation.__suppress_context__)


ValueError True


**Why this step matters.** Explicit `from error` says the new failure *is caused by* the caught failure, rather than merely occurring during its handling. Sibling handler behavior does not change.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [25]:
assert handler_events == ['inner-value-handler']
assert flow_events == ['translate', 'outer-recovery']
assert isinstance(translated.__context__, ValueError)
assert translated.__cause__ is None
assert isinstance(explicit_translation.__cause__, ValueError)
print('Problem 06: all checks passed')


Problem 06: all checks passed


### Complete explanation and design takeaway

An `except` suite is not protected by sibling `except` clauses on its own `try`. If error handling itself can fail, decide whether the failure belongs in an outer boundary or a deliberate translation, rather than assuming a second sibling will catch it.

**Your turn — extension question.** Does adding `finally` to the inner `try` cause the sibling to handle the new error?

**Reasoned answer.** No. The `finally` suite would run as control exits the inner statement, but it does not turn sibling handlers into nested handlers. An outer `try` is still required to catch the new `RuntimeError`.

---

## Problem 07 — Why errors from try-else escape that try’s except handlers

**Scenario.** A two-phase operation checks an identifier in `try`, then performs a dependent action in `else`. You must show which errors the associated `except` clause can catch.

### Contract and constraints

Record events for validation failure and post-validation failure. An `except ValueError` on this `try` must catch failures from the `try` suite but not errors newly raised inside `else`.

### Predict before running

If `else` raises `ValueError`, is it handled by the `except ValueError` belonging to the same `try` statement?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Place validation and the follow-up action in different suites

In [26]:
def two_phase(identifier):
    events = []
    try:
        if not isinstance(identifier, str):
            raise ValueError('identifier must be text')
        events.append('validated')
    except ValueError:
        events.append('caught-validation')
        return events
    else:
        events.append('entered-else')
        if identifier == 'explode':
            raise ValueError('post-validation operation failed')
        events.append('completed')
        return events

print('Bad type:', two_phase(42))
print('Healthy:', two_phase('ok'))


Bad type: ['caught-validation']
Healthy: ['validated', 'entered-else', 'completed']


**Why this step matters.** `else` runs only if the `try` suite finishes normally. Its own exceptions are *outside* the associated handlers’ catch region, even if their types happen to match.

### Step 2 — Prove that the error leaves the statement

In [27]:
post_validation_error = capture(ValueError, two_phase, 'explode')
print('Escaped error:', post_validation_error)


Escaped error: post-validation operation failed


**Why this step matters.** The error gets caught only by the external test helper, which establishes a new enclosing boundary.

### Step 3 — Choose a boundary that covers the whole operation

In [28]:
def guarded_two_phase(identifier):
    try:
        return two_phase(identifier)
    except ValueError as error:
        return ['outer-boundary', str(error)]

print(guarded_two_phase('explode'))


['outer-boundary', 'post-validation operation failed']


**Why this step matters.** An outer handler covers failures from the inner operation as a whole. Separate validation from execution when the two phases deserve different error policies.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [29]:
assert two_phase(42) == ['caught-validation']
assert two_phase('ok') == ['validated', 'entered-else', 'completed']
assert str(post_validation_error) == 'post-validation operation failed'
assert guarded_two_phase('explode')[0] == 'outer-boundary'
print('Problem 07: all checks passed')


Problem 07: all checks passed


### Complete explanation and design takeaway

`try`/`except`/`else` is a *control-flow partition*. Errors produced by `else` cannot be intercepted by the same statement’s `except` clauses. This matters when writing APIs whose post-validation phase may itself raise.

**Your turn — extension question.** Where should code that has its own independent failure policy go?

**Reasoned answer.** Give it its own `try`/`except` or wrap the entire two-phase call in a higher-level boundary. Do not expand a broad catch around unrelated work merely to make an incidental test pass.

---

## Problem 08 — A clean public traceback without lying about hidden context

**Scenario.** An API validates a regex. Its public exception should display one short failure instead of a dense compiler traceback, but debugging tools must still be able to access the original exception object.

### Contract and constraints

Use `raise InvalidPublicPattern(...) from None`. Compare the rendered traceback with the exception’s `__context__`, `__cause__`, and `__suppress_context__` fields.

### Predict before running

Does `from None` delete the regex compiler’s exception or merely hide it from default traceback formatting?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Raise a user-facing exception at the boundary

In [30]:
class InvalidPublicPattern(Exception):
    pass

def compile_public_pattern(raw):
    try:
        return re.compile(raw)
    except re.error:
        raise InvalidPublicPattern('the supplied search expression is invalid') from None

public_pattern_error = capture(InvalidPublicPattern, compile_public_pattern, '[')
print(str(public_pattern_error))


the supplied search expression is invalid


**Why this step matters.** The explicit `from None` suppresses automatic rendering of the earlier exception. It is a presentation decision; it is not a secure-erasure feature.

### Step 2 — Inspect exception internals rather than just printed output

In [31]:
print('Cause:', public_pattern_error.__cause__)
print('Context type:', type(public_pattern_error.__context__).__name__)
print('Suppress context:', public_pattern_error.__suppress_context__)


Cause: None
Context type: PatternError
Suppress context: True


**Why this step matters.** `__cause__` is `None`, while `__context__` still refers to the regex error. `__suppress_context__` requests that the context be omitted when Python prints the exception normally.

### Step 3 — Verify that the usual traceback hides the lower-level error

In [32]:
rendered_pattern = ''.join(traceback.format_exception(public_pattern_error))
print(rendered_pattern.splitlines()[-1])
print('Low-level compiler name displayed?', 're.error' in rendered_pattern)


InvalidPublicPattern: the supplied search expression is invalid
Low-level compiler name displayed? False


**Why this step matters.** Formatting does not show the preceding regex failure in the default exception chain, although programmatic access through `__context__` remains possible.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [33]:
assert public_pattern_error.__cause__ is None
assert isinstance(public_pattern_error.__context__, re.error)
assert public_pattern_error.__suppress_context__ is True
assert 'InvalidPublicPattern' in rendered_pattern
assert 're.error' not in rendered_pattern
assert compile_public_pattern(r'^ok$').fullmatch('ok') is not None
print('Problem 08: all checks passed')


Problem 08: all checks passed


### Complete explanation and design takeaway

The source notebook introduces `from None` for a cleaner displayed trace. The critical precise interpretation is: *suppress implicit context in formatting*, not *destroy previous traceback information*. If the underlying cause contains secrets, sanitize the data before putting them into exception objects at all.

**Your turn — extension question.** Would `from None` protect secrets if a monitoring system walks `error.__context__`?

**Reasoned answer.** No. Monitoring code can access the saved exception. Avoid placing sensitive data in messages, arguments, notes, or chained exception objects in the first place.

---

## Problem 09 — The right-hand side of from must also be an exception

**Scenario.** A developer writes `raise RuntimeError("failed") from "database"` to add a string explanation. Python rejects this form.

### Contract and constraints

Demonstrate the invalid cause operand and implement a correct translation with a real exception instance. Verify the original error’s identity via `__cause__`.

### Predict before running

Does `from` accept a free-form message or only an exception instance (plus the special value `None`)?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Run the invalid-cause expression inside a safe helper

In [34]:
def invalid_cause():
    raise RuntimeError('failed') from 'database'

invalid_cause_error = capture(TypeError, invalid_cause)
print(type(invalid_cause_error).__name__, invalid_cause_error)


TypeError exception causes must derive from BaseException


**Why this step matters.** The `from` operand must be an exception instance or `None`. A string is not accepted and triggers `TypeError`.

### Step 2 — Construct a real failure to carry as the cause

In [35]:
def load_missing_record(data, key):
    try:
        return data[key]
    except KeyError as error:
        raise RuntimeError(f'cannot load record {key!r}') from error

real_chain_error = capture(RuntimeError, load_missing_record, {}, 'R7')
print('Cause:', repr(real_chain_error.__cause__))


Cause: KeyError('R7')


**Why this step matters.** Here the cause is a concrete `KeyError` object representing the original failure. The new error is not merely a differently worded copy.

### Step 3 — Compare explicit cause and formatted chain wording

In [36]:
formatted_cause = ''.join(traceback.format_exception(real_chain_error))
print('Direct-cause label present?', 'direct cause' in formatted_cause)
print('Cause type:', type(real_chain_error.__cause__).__name__)


Direct-cause label present? True
Cause type: KeyError


**Why this step matters.** Normal formatting identifies explicit causes as direct causes. This differs from an implicit `__context__` chain, whose default wording says a second exception arose during handling.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [37]:
assert isinstance(invalid_cause_error, TypeError)
assert isinstance(real_chain_error.__cause__, KeyError)
assert real_chain_error.__cause__.args == ('R7',)
assert real_chain_error.__suppress_context__ is True
assert 'direct cause' in formatted_cause
assert load_missing_record({'R7': 12}, 'R7') == 12
print('Problem 09: all checks passed')


Problem 09: all checks passed


### Complete explanation and design takeaway

The `from` clause describes a relationship between **exception objects**, not a location for free-form text. Put an explanation into the new exception’s message, and link it to the actual prior exception when that cause is relevant.

**Your turn — extension question.** Is `raise RuntimeError("failed") from None` valid?

**Reasoned answer.** Yes. `None` is the one non-exception value allowed after `from`; it suppresses the display of implicit context rather than defining a cause object.

---

## Problem 10 — Capture a traceback now; replay it at a later boundary

**Scenario.** A diagnostic engine must store a failure and later surface a *new public exception* with the original recorded traceback frames attached.

### Contract and constraints

Capture an original `OSError`, save its `__traceback__`, and later use `raise NewError(...).with_traceback(saved_tb) from original`. Distinguish the new traceback from the explicit causal link.

### Predict before running

Does `from original` alone attach original frames to the new exception’s own `__traceback__`?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Capture original error and traceback as separate data

In [38]:
def fail_in_sensor():
    raise OSError('sensor offline')

sensor_original = capture(OSError, fail_in_sensor)
saved_sensor_tb = sensor_original.__traceback__
print('Original frame names:', [f.name for f in traceback.extract_tb(saved_sensor_tb)])


Original frame names: ['capture', 'fail_in_sensor']


**Why this step matters.** An exception object has its own traceback. Saving it before later processing gives a concrete traceback object that can be deliberately attached elsewhere.

### Step 2 — Raise a new public error with two independent links

In [39]:
class SensorUnavailable(Exception):
    pass

def replay_sensor_failure(original, stored_tb):
    raise SensorUnavailable('cannot read temperature').with_traceback(stored_tb) from original

sensor_public = capture(SensorUnavailable, replay_sensor_failure, sensor_original, saved_sensor_tb)
print('Cause preserved:', sensor_public.__cause__ is sensor_original)


Cause preserved: True


**Why this step matters.** `.with_traceback(stored_tb)` chooses the public exception’s starting traceback; `from original` specifies the explicit causal exception. The new `raise` also adds its current frame.

### Step 3 — Inspect the frames of both exception objects

In [40]:
public_sensor_frames = [f.name for f in traceback.extract_tb(sensor_public.__traceback__)]
original_sensor_frames = [f.name for f in traceback.extract_tb(sensor_original.__traceback__)]
print('Public frames:', public_sensor_frames)
print('Cause frames:', original_sensor_frames)


Public frames: ['capture', 'replay_sensor_failure', 'capture', 'fail_in_sensor']
Cause frames: ['capture', 'fail_in_sensor']


**Why this step matters.** The public error includes historical frames as well as frames from the later raise. The cause has its own traceback separately. Replaying old frames is a specialized diagnostic choice, not a routine translation pattern.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [41]:
assert sensor_public.__cause__ is sensor_original
assert 'fail_in_sensor' in public_sensor_frames
assert 'replay_sensor_failure' in public_sensor_frames
assert 'fail_in_sensor' in original_sensor_frames
assert isinstance(saved_sensor_tb, type(sensor_original.__traceback__))
print('Problem 10: all checks passed')


Problem 10: all checks passed


### Complete explanation and design takeaway

Cause links and traceback attachment are different mechanisms. Ordinary API translation usually needs `raise ... from error` only. `.with_traceback` is appropriate only when your use case explicitly requires transplanting saved frames, and the new raise still adds a frame.

**Your turn — extension question.** Can you infer the entire visible formatted chain by looking at `sensor_public.__traceback__` alone?

**Reasoned answer.** No. A complete formatted exception chain may follow `__cause__` (and, when appropriate, context), each with its own traceback. Examining one traceback attribute shows only that exception’s traceback.

---

# Part III — Advanced propagation across language features

These are **extensions beyond the supplied lesson**. The same ideas—what is raised, what is caught, where the boundary sits, and which cause survives—become especially important in generators, asynchronous programs, and worker threads.

## Problem 11 — The surprising transformation of StopIteration inside a generator

**Scenario.** A developer raises `StopIteration` inside a generator to indicate an early return. Modern Python converts this accidental internal failure into a `RuntimeError` (PEP 479).

### Contract and constraints

Contrast an explicit `raise StopIteration` within a generator with a correct `return value` and inspect the converted exception’s cause.

### Predict before running

Will a caller see plain `StopIteration` or `RuntimeError` when generator code explicitly raises `StopIteration` during iteration?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Build the intentionally broken generator

In [42]:
def broken_generator():
    yield 'first'
    raise StopIteration('do not do this inside generator code')

broken_iter = broken_generator()
print(next(broken_iter))


first


**Why this step matters.** A generator body runs lazily. The invalid internal `raise` has not yet executed when the iterator is created or when the first value is yielded.

### Step 2 — Resume and inspect the transformed failure

In [43]:
generator_failure = capture(RuntimeError, next, broken_iter)
print('Visible type:', type(generator_failure).__name__)
print('Cause type:', type(generator_failure.__cause__).__name__)


Visible type: RuntimeError
Cause type: StopIteration


**Why this step matters.** Python transforms a `StopIteration` escaping from inside generator code into `RuntimeError`, preserving the original `StopIteration` as its cause. This prevents accidental premature termination from silently hiding bugs.

### Step 3 — Use a generator return value correctly

In [44]:
def correct_generator():
    yield 'first'
    return 'completed normally'

correct_iter = correct_generator()
print(next(correct_iter))
normal_stop = capture(StopIteration, next, correct_iter)
print('Generator return value:', normal_stop.value)


first
Generator return value: completed normally


**Why this step matters.** `return` terminates the generator properly. The generator protocol represents final completion as `StopIteration`, and its `.value` carries the returned value. That is different from manually raising `StopIteration` in the generator body.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [45]:
assert isinstance(generator_failure.__cause__, StopIteration)
assert 'generator raised StopIteration' in str(generator_failure)
assert normal_stop.value == 'completed normally'
assert list(correct_generator()) == ['first']
print('Problem 11: all checks passed')


Problem 11: all checks passed


### Complete explanation and design takeaway

`raise` interacts with language protocols. Within generator bodies, explicit `StopIteration` can be a bug because it is transformed into `RuntimeError`. Use `return` for natural completion; reserve other exception types for genuine failures.

**Your turn — extension question.** Does `list(correct_generator())` expose the generator’s final return value?

**Reasoned answer.** No. `list` consumes yielded items; the return value is attached to the terminal `StopIteration.value`, which normal iteration hides from callers.

---

## Problem 12 — Translate task failures without swallowing asyncio cancellation

**Scenario.** An async adapter fetches a key and translates a missing key into a public domain failure. Cancellation must propagate; it must not be presented as a missing key.

### Contract and constraints

Write an async function with an explicit cause for `KeyError`. Demonstrate cancellation separately, and show why catching only `KeyError` keeps unrelated task control signals outside this policy.

### Predict before running

What should happen if a caller cancels a pending task before it resumes?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Create a small async function with one translation boundary

In [46]:
class RemoteItemMissing(Exception):
    pass

async def async_fetch_item(store, key):
    await asyncio.sleep(0)
    try:
        return store[key]
    except KeyError as error:
        raise RemoteItemMissing(f'item {key!r} not found') from error

print('Defined async API')


Defined async API


**Why this step matters.** The narrow `KeyError` handler translates data lookup failures only. Other failures continue to propagate rather than being relabeled as a missing item.

### Step 2 — Execute both ordinary and failing task paths

In [47]:
async def run_async_examples():
    good = await async_fetch_item({'item': 99}, 'item')
    try:
        await async_fetch_item({}, 'absent')
    except RemoteItemMissing as error:
        return good, error
    raise AssertionError('Expected RemoteItemMissing')

async_good, async_missing = await run_async_examples()
print('Value:', async_good, '| cause:', type(async_missing.__cause__).__name__)


Value: 99 | cause: KeyError


**Why this step matters.** An exception raised inside a coroutine is delivered when the coroutine is awaited. Explicit cause information travels with the exception object.

### Step 3 — Cancel a task and observe the distinct signal

In [48]:
async def demonstrate_cancellation():
    task = asyncio.create_task(async_fetch_item({}, 'never-run'))
    task.cancel()
    try:
        await task
    except asyncio.CancelledError as error:
        return error
    raise AssertionError('Cancellation should propagate')

cancellation_signal = await demonstrate_cancellation()
print('Cancelled:', type(cancellation_signal).__name__)


Cancelled: CancelledError


**Why this step matters.** Cancellation is not a lookup failure and must not be translated into one. In modern Python `asyncio.CancelledError` inherits directly from `BaseException`, another reason not to intercept everything blindly.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [49]:
assert async_good == 99
assert isinstance(async_missing.__cause__, KeyError)
assert isinstance(cancellation_signal, asyncio.CancelledError)
assert not isinstance(cancellation_signal, RemoteItemMissing)
assert issubclass(asyncio.CancelledError, BaseException)
print('Problem 12: all checks passed')


Problem 12: all checks passed


### Complete explanation and design takeaway

The same `raise ... from error` idiom works in asynchronous code. The important new issue is **exception policy**: translate only the failure your layer owns and allow cancellation to retain its special meaning.

**Your turn — extension question.** Why is `except BaseException: raise RemoteItemMissing(...)` particularly harmful here?

**Reasoned answer.** It could turn cancellation and process-control exceptions into misleading domain errors, preventing callers from interpreting them correctly. Narrow, explicit handling is the safer contract.

---

## Problem 13 — A worker thread raises; the Future delivers the same failure

**Scenario.** A CPU-bound job runs in a `ThreadPoolExecutor`. Its `Future.result()` must raise the worker failure rather than return a fake success value.

### Contract and constraints

Raise a custom exception in the worker. Retrieve it via `Future.result()` and inspect worker traceback frames. Then show a successful result through the same API.

### Predict before running

Does `future.result()` silently return `None` after a worker raises an exception?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Define an exception and a worker with two outcomes

In [50]:
class ComputationRejected(Exception):
    pass

def compute_square(value):
    if value < 0:
        raise ComputationRejected('negative input is not allowed')
    return value * value

print('Worker defined')


Worker defined


**Why this step matters.** The worker owns the domain validation and raises from inside the thread. Thread pool machinery stores the failure for later retrieval by the calling thread.

### Step 2 — Observe a failure via Future.result

In [51]:
with ThreadPoolExecutor(max_workers=1) as executor:
    rejected_future = executor.submit(compute_square, -2)
    worker_error = capture(ComputationRejected, rejected_future.result)
    healthy_future = executor.submit(compute_square, 7)
    healthy_square = healthy_future.result()

print('Failure:', worker_error)
print('Success:', healthy_square)


Failure: negative input is not allowed
Success: 49


**Why this step matters.** `Future.result()` re-raises the recorded worker failure on the consumer side. The success path returns the function’s computed value.

### Step 3 — Inspect both worker and retrieval traceback locations

In [52]:
future_frame_names = [frame.name for frame in traceback.extract_tb(worker_error.__traceback__)]
print('Representative frames:', future_frame_names[-6:])


Representative frames: ['capture', 'result', '__get_result', 'run', 'compute_square']


**Why this step matters.** The traceback includes the worker computation frame and retrieval machinery. This is useful evidence of where an exception originated versus where it surfaced.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [53]:
assert isinstance(worker_error, ComputationRejected)
assert rejected_future.exception() is worker_error
assert 'compute_square' in future_frame_names
assert healthy_square == 49
print('Problem 13: all checks passed')


Problem 13: all checks passed


### Complete explanation and design takeaway

Asynchronous execution and worker threads change the point at which the caller *observes* a raised exception, not its public failure contract. Do not hide worker failures behind `None` or a success-shaped value unless the API explicitly promises that behavior.

**Your turn — extension question.** How do you inspect a worker failure without calling `.result()`?

**Reasoned answer.** Once the Future is done, `.exception()` returns the exception object (or `None` if successful). Note that `.exception()` itself can raise if the Future was cancelled, so cancellation remains its own case.

---

## Problem 14 — Split a nested ExceptionGroup without losing structural context

**Scenario.** Several independent checks fail; one group contains a nested group. You need to separately handle value failures and type failures without flattening away all group structure.

### Contract and constraints

Construct a nested `ExceptionGroup`. Use `except* ValueError` and `except* TypeError` in separate clauses; recursively collect leaf exception types for test assertions.

### Predict before running

Does `except* ValueError` receive just a bare `ValueError`, or an exception group containing matched leaves?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Build a nested group of distinct failures

In [54]:
multi_failure = ExceptionGroup('checks failed', [
    ValueError('negative balance'),
    ExceptionGroup('nested input checks', [
        TypeError('missing string'),
        ValueError('wrong date'),
    ]),
])
print('Top-level members:', len(multi_failure.exceptions))


Top-level members: 2


**Why this step matters.** An exception group is one raised object holding multiple exceptions and potentially nested groups. This example represents independent failures found in one batch.

### Step 2 — Select subgroups with except-star clauses

In [55]:
selected_groups = {}
try:
    raise multi_failure
except* ValueError as matched_values:
    selected_groups['values'] = matched_values
except* TypeError as matched_types:
    selected_groups['types'] = matched_types

print('Matched subgroup types:', type(selected_groups['values']).__name__, type(selected_groups['types']).__name__)


Matched subgroup types: ExceptionGroup ExceptionGroup


**Why this step matters.** Each `except*` gets a subgroup retaining relevant structure, not a single leaf exception. The two clauses partition matching leaves by exception type.

### Step 3 — Write a traversal for independent verification

In [56]:
def group_leaves(error):
    if isinstance(error, BaseExceptionGroup):
        for child in error.exceptions:
            yield from group_leaves(child)
    else:
        yield error

value_messages = [str(e) for e in group_leaves(selected_groups['values'])]
type_messages = [str(e) for e in group_leaves(selected_groups['types'])]
print('Values:', value_messages)
print('Types:', type_messages)


Values: ['negative balance', 'wrong date']
Types: ['missing string']


**Why this step matters.** Recursing through the group separates the leaf failures needed for checks. In a production error reporter, retaining group structure can reveal which subsystem or batch generated each leaf.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [57]:
assert sorted(value_messages) == ['negative balance', 'wrong date']
assert type_messages == ['missing string']
assert all(isinstance(e, ValueError) for e in group_leaves(selected_groups['values']))
assert all(isinstance(e, TypeError) for e in group_leaves(selected_groups['types']))
print('Problem 14: all checks passed')


Problem 14: all checks passed


### Complete explanation and design takeaway

`except*` performs type-based matching inside groups and passes a group-shaped result to its handler. This is more powerful than a conventional `except`, but also requires clear policies for unmatched leaves; otherwise they continue propagating.

**Your turn — extension question.** What if the original group also held `OSError`, but no `except* OSError` handled it?

**Reasoned answer.** The unmatched OSError subgroup would propagate after matched subgroups had been handled. Add a suitable handler only if your application can genuinely address that failure.

---

# Part IV — Contracts, tests, and algorithmic failure design

These problems integrate raised exceptions into contracts that other programmers can rely on. We focus on precise assertions, inheritance, and fail-fast algorithms, then finish with two multi-layer capstones.

## Problem 15 — Test the exception contract with unittest, not printouts

**Scenario.** An email-normalization API must reject invalid objects predictably. Your test suite should examine types, messages, normal returns, and exception metadata—not just whether a red traceback appeared.

### Contract and constraints

Implement `canonical_email` and at least four `unittest` methods using `assertRaises`, `assertRaisesRegex`, and `assertEqual`. Prove the negative cases and the healthy path.

### Predict before running

How can a unit test fail if the function unexpectedly returns instead of raising?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Define a deterministic, deliberately modest normalization policy

In [58]:
class EmailFormatError(ValueError):
    pass

def canonical_email(value):
    if not isinstance(value, str):
        raise TypeError('email must be a string')
    text = value.strip()
    if text.count('@') != 1:
        raise EmailFormatError('email must contain exactly one @')
    local, domain = text.split('@')
    if not local or not domain or '.' not in domain:
        raise EmailFormatError('invalid local part or domain')
    return local + '@' + domain.lower()

print(canonical_email(' User@EXAMPLE.COM '))


User@example.com


**Why this step matters.** This is a *toy normalization contract*, not a complete implementation of email standards. The lesson concerns precise failure behavior and reproducible tests.

### Step 2 — Write tests that distinguish success, type, and format failures

In [59]:
class EmailContractTests(unittest.TestCase):
    def test_normalization(self):
        self.assertEqual(canonical_email(' A@EXAMPLE.COM '), 'A@example.com')

    def test_wrong_type(self):
        with self.assertRaises(TypeError) as caught:
            canonical_email(None)
        self.assertIn('string', str(caught.exception))

    def test_missing_separator(self):
        with self.assertRaisesRegex(EmailFormatError, 'exactly one'):
            canonical_email('invalid')

    def test_missing_domain(self):
        with self.assertRaises(EmailFormatError) as caught:
            canonical_email('user@')
        self.assertIsInstance(caught.exception, ValueError)

print('Defined four exception-contract tests')


Defined four exception-contract tests


**Why this step matters.** `assertRaises` fails the test automatically if no matching exception is raised. The context manager gives access to the captured object for deeper assertions. `assertRaisesRegex` also checks the message.

### Step 3 — Run tests without terminating the notebook on failures

In [60]:
email_suite = unittest.defaultTestLoader.loadTestsFromTestCase(EmailContractTests)
email_result = unittest.TextTestRunner(verbosity=1).run(email_suite)
print('Tests run:', email_result.testsRun, '| successful:', email_result.wasSuccessful())


....
----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK


Tests run: 4 | successful: True


**Why this step matters.** A repeatable test suite turns the exception policy into executable documentation, rather than relying on manually inspecting a sequence of print statements.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [61]:
assert email_result.wasSuccessful()
assert email_result.testsRun == 4
assert issubclass(EmailFormatError, ValueError)
assert isinstance(capture(EmailFormatError, canonical_email, 'a@@b.com'), ValueError)
print('Problem 15: all checks passed')


Problem 15: all checks passed


### Complete explanation and design takeaway

Tests should assert both **which error** is raised and **when** it is raised. Use stable message fragments or structured attributes rather than asserting a full, brittle interpreter-generated traceback. Good tests also exercise valid inputs.

**Your turn — extension question.** Why should this notebook not advertise `canonical_email` as a standards-compliant validator?

**Reasoned answer.** Real email syntax is much more complicated. This function intentionally specifies a small teaching contract; broader claims would require a different design, reference standard, and tests.

---

## Problem 16 — Design a dependency checker with different exceptions for unknown nodes and cycles

**Scenario.** A build planner has tasks with dependency edges. An unknown dependency is an input error; a cycle is a graph-structure error. Return a topological ordering only when the graph is valid.

### Contract and constraints

Define domain exceptions, use depth-first search and a recursion stack, and raise a cycle exception containing the path. Ensure incomplete results are never returned after failure.

### Predict before running

For A → B → C → A, which nodes should be recorded in the cycle? How is an unknown node different?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Create different error types and path metadata

In [62]:
class DependencyError(Exception):
    pass

class UnknownTask(DependencyError):
    def __init__(self, task, dependency):
        self.task, self.dependency = task, dependency
        super().__init__(f'{task!r} depends on unknown task {dependency!r}')

class CyclicDependency(DependencyError):
    def __init__(self, cycle):
        self.cycle = tuple(cycle)
        super().__init__('cycle: ' + ' -> '.join(self.cycle))

print('Domain hierarchy ready')


Domain hierarchy ready


**Why this step matters.** Both failures inherit from a shared library-level base, but retain different structured information. Callers can catch `DependencyError` broadly or each subclass specifically.

### Step 2 — Build a depth-first topological sorter with two raise sites

In [63]:
def order_tasks(graph):
    visiting = []
    done = set()
    ordered = []

    def visit(task):
        if task in done:
            return
        if task in visiting:
            start = visiting.index(task)
            raise CyclicDependency(visiting[start:] + [task])
        visiting.append(task)
        for dependency in graph[task]:
            if dependency not in graph:
                raise UnknownTask(task, dependency)
            visit(dependency)
        visiting.pop()
        done.add(task)
        ordered.append(task)

    for task in graph:
        visit(task)
    return ordered

print(order_tasks({'compile': ['fetch'], 'fetch': []}))


['fetch', 'compile']


**Why this step matters.** The two invalid structures are detected at their natural discovery sites: unknown edge targets during edge traversal and cycles when revisiting an active recursion path. The function returns only after traversal completes successfully.

### Step 3 — Inspect both failures instead of only printing a traceback

In [64]:
cycle_found = capture(CyclicDependency, order_tasks, {'A': ['B'], 'B': ['C'], 'C': ['A']})
unknown_found = capture(UnknownTask, order_tasks, {'A': ['absent']})
print('Cycle path:', cycle_found.cycle)
print('Unknown edge:', unknown_found.task, '->', unknown_found.dependency)


Cycle path: ('A', 'B', 'C', 'A')
Unknown edge: A -> absent


**Why this step matters.** The cycle path ends with the repeated starting node; unknown-task metadata points directly to the offending edge. These properties are more reliable than slicing message text.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [65]:
assert cycle_found.cycle == ('A', 'B', 'C', 'A')
assert unknown_found.dependency == 'absent'
valid_order = order_tasks({'ship': ['test'], 'test': ['build'], 'build': []})
assert valid_order.index('build') < valid_order.index('test') < valid_order.index('ship')
assert isinstance(cycle_found, DependencyError)
assert isinstance(unknown_found, DependencyError)
print('Problem 16: all checks passed')


Problem 16: all checks passed


### Complete explanation and design takeaway

Good exception design exposes **semantically different failure modes** as different types, with useful data. A fail-fast graph algorithm should never return a partial ordering as if it were complete. This scenario applies the same `raise` fundamentals to recursive algorithms.

**Your turn — extension question.** What should a caller do if it wants to display all unknown tasks, not just the first?

**Reasoned answer.** Use a separate validation pass to collect unknown edges before sorting, potentially reporting them together. The fail-fast implementation intentionally raises the first discovered error and therefore does not promise exhaustive diagnostics.

---

## Problem 17 — Capstone A: a template compiler with three layers of failures

**Scenario.** An application renders placeholders such as `{name}`. There are three distinct boundaries: bad template syntax, unsupported placeholder names, and missing values at render time.

### Contract and constraints

Build a strict parser for `{identifier}` and literal text. Translate an internal `KeyError` with explicit cause; raise syntax/name errors directly. The renderer must never emit a misleading partially rendered result on failure.

### Predict before running

For `"Hello {user}!"`, which failures belong to compilation versus rendering? What should happen if values lack `"user"`?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Specify a domain hierarchy and a strict token grammar

In [66]:
class TemplateFailure(Exception):
    pass

class TemplateSyntaxFailure(TemplateFailure):
    pass

class TemplateNameFailure(TemplateFailure):
    pass

class MissingTemplateValue(TemplateFailure):
    pass

TEMPLATE_NAME = re.compile(r'[A-Za-z_][A-Za-z0-9_]*\Z')
print('Allowed placeholder example:', bool(TEMPLATE_NAME.fullmatch('user_2')))


Allowed placeholder example: True


**Why this step matters.** Distinct exception classes let callers choose specific recovery actions. A simple identifier grammar makes the teaching parser predictable; it is deliberately not a full template engine.

### Step 2 — Compile the template into literal and placeholder tokens

In [67]:
def compile_template(text):
    if not isinstance(text, str):
        raise TypeError('template must be text')
    tokens = []
    cursor = 0
    while cursor < len(text):
        opening = text.find('{', cursor)
        unmatched_close = text.find('}', cursor)
        if unmatched_close != -1 and (opening == -1 or unmatched_close < opening):
            raise TemplateSyntaxFailure(f'unexpected }} at offset {unmatched_close}')
        if opening == -1:
            tokens.append(('literal', text[cursor:]))
            break
        tokens.append(('literal', text[cursor:opening]))
        closing = text.find('}', opening + 1)
        if closing == -1:
            raise TemplateSyntaxFailure(f'unclosed {{ at offset {opening}')
        name = text[opening + 1:closing]
        if not TEMPLATE_NAME.fullmatch(name):
            raise TemplateNameFailure(f'invalid placeholder name: {name!r}')
        tokens.append(('name', name))
        cursor = closing + 1
    return tokens

sample_tokens = compile_template('Hello {user}!')
print(sample_tokens)


[('literal', 'Hello '), ('name', 'user'), ('literal', '!')]


**Why this step matters.** Compilation validates structure before any values are substituted. The implementation explicitly rejects unmatched closing braces, unclosed openings, and invalid names. Literal text is represented separately from placeholders.

### Step 3 — Render only after values are available and all tokens are valid

In [68]:
def render_template(template, values):
    if not isinstance(values, dict):
        raise TypeError('values must be a dictionary')
    tokens = compile_template(template)
    pieces = []
    for kind, data in tokens:
        if kind == 'literal':
            pieces.append(data)
        else:
            try:
                replacement = values[data]
            except KeyError as error:
                raise MissingTemplateValue(f'no value supplied for {data!r}') from error
            pieces.append(str(replacement))
    return ''.join(pieces)

print(render_template('Hello {user}!', {'user': 'Ada'}))


Hello Ada!


**Why this step matters.** The missing-value boundary maps a dictionary `KeyError` to a domain-specific error with its cause preserved. A partial `pieces` list is local and is never returned when an exception is raised.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [69]:
assert render_template('X={x}; Y={y}', {'x': 2, 'y': 3}) == 'X=2; Y=3'
assert render_template('plain text', {}) == 'plain text'
assert isinstance(capture(TemplateSyntaxFailure, compile_template, 'oops }'), TemplateSyntaxFailure)
assert isinstance(capture(TemplateSyntaxFailure, compile_template, 'oops {name'), TemplateSyntaxFailure)
assert isinstance(capture(TemplateNameFailure, compile_template, '{bad-name}'), TemplateNameFailure)
missing_value = capture(MissingTemplateValue, render_template, 'Hi {user}', {})
assert isinstance(missing_value.__cause__, KeyError)
assert missing_value.__cause__.args == ('user',)
print('Problem 17: all checks passed')


Problem 17: all checks passed


### Complete explanation and design takeaway

The capstone separates **compile-time** failures from **render-time** failures and preserves the original lookup error where useful. A clear exception hierarchy and local output assembly prevent malformed input from producing success-shaped, partially rendered output.

**Your turn — extension question.** Should this minimal compiler accept escaped braces such as `{{`?

**Reasoned answer.** Not under its present grammar; it would reject them as an invalid placeholder name. Supporting escaped braces requires an explicit tokenization rule and new tests, rather than silently treating ambiguous syntax as valid.

---

## Problem 18 — Capstone B: strict binary-frame parsing and causal diagnostics

**Scenario.** A device frame has exactly 2 magic bytes (`b"PX"`), a 2-byte unsigned big-endian payload length, a payload, and a 1-byte checksum equal to `sum(payload) % 256`.

### Contract and constraints

Implement encode/decode. Translate the low-level `struct.error` from a short header with an explicit cause. Raise independent framing errors directly for bad magic, truncated payload, extra bytes, or checksum mismatch.

### Predict before running

What is the difference between a header too short for `struct.unpack` and a complete frame with the wrong checksum?

*Work it out before expanding the solution. The next cells give the answer in pieces.*

### Step 1 — Define the frame vocabulary and encoder

In [70]:
class FrameError(Exception):
    pass

class FrameHeaderError(FrameError):
    pass

class FramePayloadError(FrameError):
    pass


def encode_frame(payload):
    if not isinstance(payload, bytes):
        raise TypeError('payload must be bytes')
    if len(payload) > 65535:
        raise ValueError('payload exceeds uint16 capacity')
    return b'PX' + struct.pack('>H', len(payload)) + payload + bytes([sum(payload) % 256])

encoded_example = encode_frame(b'ABC')
print('Encoded frame:', encoded_example)


Encoded frame: b'PX\x00\x03ABC\xc6'


**Why this step matters.** A framing error taxonomy separates structural header errors from payload failures. The encoder provides a reliable valid frame for round-trip testing.

### Step 2 — Parse the header with narrow causal translation

In [71]:
def decode_frame(frame):
    if not isinstance(frame, bytes):
        raise TypeError('frame must be bytes')
    if frame[:2] != b'PX':
        raise FrameHeaderError('incorrect magic marker')
    try:
        expected_length = struct.unpack('>H', frame[2:4])[0]
    except struct.error as error:
        raise FrameHeaderError('header length field is truncated') from error
    expected_total = 2 + 2 + expected_length + 1
    if len(frame) != expected_total:
        raise FramePayloadError(f'frame size mismatch: expected {expected_total}, got {len(frame)}')
    payload = frame[4:-1]
    if frame[-1] != sum(payload) % 256:
        raise FramePayloadError('checksum mismatch')
    return payload

print('Decoded:', decode_frame(encoded_example))


Decoded: b'ABC'


**Why this step matters.** The parser maps only the exact `struct.error` caused by decoding an incomplete two-byte length field. Magic, size, and checksum failures are explicit protocol violations and do not need invented low-level causes.

### Step 3 — Exercise malformed input at each protocol layer

In [72]:
short_header = capture(FrameHeaderError, decode_frame, b'PX\x00')
bad_magic = capture(FrameHeaderError, decode_frame, b'NO\x00\x00\x00')
short_payload = capture(FramePayloadError, decode_frame, encoded_example[:-1])
bad_checksum_bytes = bytearray(encoded_example)
bad_checksum_bytes[-1] ^= 1
bad_checksum = capture(FramePayloadError, decode_frame, bytes(bad_checksum_bytes))
print('Truncated header cause:', type(short_header.__cause__).__name__)
print('Bad checksum cause:', bad_checksum.__cause__)


Truncated header cause: error
Bad checksum cause: None


**Why this step matters.** The original unpacking error is relevant to a truncated header; checksum mismatch is a directly recognized domain failure. Every invalid call is captured without aborting later tests.

### Verify the solution: executable edge-case checks

These assertions document the intended API contract. A green run is evidence for the particular behaviors checked; it is not a proof about every possible input.

In [73]:
assert isinstance(short_header.__cause__, struct.error)
assert bad_magic.__cause__ is None
assert short_payload.__cause__ is None
assert bad_checksum.__cause__ is None
for payload in (b'', b'hello', bytes(range(256))):
    assert decode_frame(encode_frame(payload)) == payload
assert isinstance(capture(TypeError, decode_frame, 'text'), TypeError)
assert isinstance(capture(TypeError, encode_frame, bytearray(b'A')), TypeError)
print('Problem 18: all checks passed')


Problem 18: all checks passed


### Complete explanation and design takeaway

Protocol parsing reveals the benefit of precise exception contracts: syntax/structure, length, and checksum are separate facts, and only genuine lower-level failures should be explicitly chained. The round-trip checks confirm normal behavior; invalid-frame checks confirm the advertised failure vocabulary.

**Your turn — extension question.** Why not catch every `Exception` in `decode_frame` and raise `FrameError("invalid")`?

**Reasoned answer.** Doing so would erase distinctions between malformed frames, wrong input types, and programming defects. It would also conceal where the parser actually failed. Narrow translation, direct domain raises, and structured tests make failures actionable.

---

# Final review — solve without looking

Try to answer in complete sentences before checking the concise answer key below.

1. A custom exception requires constructor arguments. What happens if you `raise CustomError` without supplying them?
2. Why should validation happen before assignment in a property setter?
3. What does `raise PublicError(...) from original` control, and what does it **not** control?
4. Can a sibling `except` catch an exception newly raised from an earlier sibling handler?
5. Why does `raise ... from None` not guarantee secret removal?
6. How does a generator correctly return a terminal value?
7. What makes async cancellation different from a missing-key failure?
8. What is passed to `except* ValueError` when the raised value is an exception group?
9. Should every protocol-validation failure get a synthetic low-level cause?
10. What must a caller check besides exception type when verifying API behavior?

### Concise answer key

1. Python tries to instantiate the class without arguments; its constructor may raise `TypeError`.
2. A rejected update must leave the previously valid state unchanged.
3. It records the explicit `__cause__` and controls chain presentation, but it does not transplant the original traceback into the new error.
4. No. Put the handler at an enclosing boundary or use nested `try` statements.
5. The previous exception can remain accessible through `__context__`; do not put secrets into diagnostic data at all.
6. Use `return value`; the terminal `StopIteration.value` stores the generator’s return value.
7. Cancellation is a control signal, not a domain lookup failure; let it propagate.
8. A matched subgroup, not a bare `ValueError` leaf.
9. No. Directly detected violations should be raised directly; chain genuine lower-level exceptions when they add diagnostic value.
10. Check normal results, message or structured attributes, cause/context where relevant, and state integrity after rejected calls.

In [74]:
# One final cumulative, executable notebook-wide smoke check.
assert read_timeout({'service': {'timeout': '20'}}) == 20
assert require_match('ok', 'okay') == 'ok'
assert isinstance(public_pattern_error.__context__, re.error)
assert isinstance(generator_failure.__cause__, StopIteration)
assert worker_error is rejected_future.exception()
assert email_result.wasSuccessful()
assert render_template('{x}', {'x': 'done'}) == 'done'
assert decode_frame(encode_frame(b'final')) == b'final'
print('FINAL CHECK: all tutorial examples and representative contracts completed successfully.')


FINAL CHECK: all tutorial examples and representative contracts completed successfully.
